# Necessary Imports

In [1]:
import numpy as np
import pandas as pd

In [2]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service

In [3]:
from selenium.webdriver.common.by import By
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver.support.ui import WebDriverWait
import regex as re
from bs4 import BeautifulSoup
import time

# Getting Product Links from a Category Page

In [4]:
# To open a new window: webdriver.Chrome(here pass the driver as a service)
# we need to create a object of a service class so that we can pass it to open a chrome window 
s=Service("C:/Users/Tanvir Ahmed/Downloads/chromedriver-win64/chromedriver.exe")
driver = webdriver.Chrome(service=s)
#after opening the window, searching for the link
driver.get('https://www.zazzle.com/s/photopop+graduation+invitations?st=orderitemcount_month')
time.sleep(2) #wait 2 seconds
driver.execute_script("window.scrollTo(0, document.body.scrollHeight)") #scroll to the bottom to load all product
time.sleep(3)
page=driver.page_source #getting the source code of that loaded page
soup = BeautifulSoup(page,'lxml') #create a soup object to parse it, to get all products link
driver.quit() #close the window

In [5]:
links=soup.find_all('a',{'class':'Link Link--marketplaceTheme SearchResultsGridCell2_link'}) #getting all product links available in the page by selecting specific class
link=[]
for i in range(0,len(links),2): #there are double link for a single product scrapped, that's why I just took every alternatinve links
    link.append(links[i]['href']) #getting the only links
link

['https://www.zazzle.com/photopop_modern_arch_photo_graduation_party_invitation-256002669626538952',
 'https://www.zazzle.com/modern_photopop_collage_graduation_party_invitation-256673808092573475',
 'https://www.zazzle.com/modern_photopop_graduation_party_invitation-256192810979022421',
 'https://www.zazzle.com/modern_photopop_graduation_party_invitation-256504032064049343',
 'https://www.zazzle.com/modern_photopop_graduation_announcement_party-256111195747049891',
 'https://www.zazzle.com/bold_grad_photopop_graduation_announcement-256319453709037882',
 'https://www.zazzle.com/soft_editorial_grad_photopop_graduation_invitation-256920148295606443',
 'https://www.zazzle.com/elegant_photopop_graduation_announcement_party-256827312935044873',
 'https://www.zazzle.com/crushed_it_photopop_graduation_party_invitation-256318071906610425',
 'https://www.zazzle.com/contemporary_photopop_graduation_party_invitation-256660884817538404',
 'https://www.zazzle.com/modern_vertical_photopop_graduation

# Scraping Data From Each Prodcut

In [ ]:
s=Service("C:/Users/Tanvir Ahmed/Downloads/chromedriver-win64/chromedriver.exe")
driver = webdriver.Chrome(service=s)

title=[]
view=[]
created_date=[]
tag=[]
picture_link = []

wait = WebDriverWait(driver, 10) #set a waiter

for i in link:
    driver.get(i) #visiting each link

    # wait until page loads the title
    try:
        wait.until(EC.presence_of_element_located((By.CLASS_NAME, "ProductSpaceDetailsPod_title")))
    except:
        print(f"Page not loaded properly: {i}")
        continue
    #scroll to the end of the page
    driver.execute_script("window.scrollTo(0, document.body.scrollHeight)")
    
    #wait until page loads the OtherInfo block
    try:
        wait.until(EC.presence_of_element_located((By.CLASS_NAME, "OtherInfo")))
    except:
        print(f"Page not loaded properly: {i}")
        continue    

    #after loead all information get the source html of the page
    html=driver.page_source

    #now I can convert it to a soup object and then parse it to get all information
    soup=BeautifulSoup(html,'lxml')

    #get the image link
    try:
        img = soup.find('div', {'class': 'ProductView-imageContainer'}).find('img')
        srcset = img.get('srcset')
        img_link = srcset.split(",")[-1].split()[0]  #get the exact link
    except:
        img_link = None
    picture_link.append(img_link)

    #get the title
    try:
        t = soup.find('h1', {'class': 'ProductSpaceDetailsPod_title'}).text.strip()
    except:
        t = None
    title.append(t)

    #get the total views
    try:
        v = soup.find('div', {'class': 'BehavioralCallout_copy'}).find_all('div')[1].text.strip()
    except:
        v = None
    view.append(v)

    #get the created on date
    try:
        d = soup.find('div', {'class': 'OtherInfo'}).find_all('div')[1].find('span').text.strip()
    except:
        d = None
    created_date.append(d)

    #get all tags and merge them into a string
    try:
        all_tags = soup.find('div', {'class': 'Tags-tagsList'}).find_all('span')
        s = [tg.text.strip() for tg in all_tags]
        s = ", ".join(s)
    except:
        s = None
    tag.append(s)

#close browser
driver.quit()

#store those data into a dataframe
df=pd.DataFrame({
    "Link":link,
    "Image":picture_link,
    'Title':title,
    'View':view,
    'Created Date':created_date,
    'Tag':tag
})


In [18]:
#making backups
df_original=df.copy()

# Data Cleaning

In [19]:
#converting the date to datetime format and sort the dataframe on date ( resent product comes first)
df["Created Date"] = pd.to_datetime(
    df["Created Date"],
    format="%m/%d/%Y, %I:%M %p",
    errors="coerce"
).dt.date
df = df.sort_values(by="Created Date", ascending=False)
df

,Image,Link,Title,View,Created Date,Tag
24,https://rlv.zcache.com/photopop_modern_arch_ph...,https://www.zazzle.com/photopop_modern_arch_ph...,PhotoPop Modern Arch Photo Graduation Party Fo...,265 people viewed this design,2026-02-11,"graduation announcement template, photo gradua..."
10,https://rlv.zcache.com/modern_vertical_photopo...,https://www.zazzle.com/modern_vertical_photopo...,Modern Vertical PhotoPop Graduation Party Foil...,169 people viewed this design,2026-02-11,"graduation announcement, photopop graduation, ..."
23,https://rlv.zcache.com/photopop_minimal_year_o...,https://www.zazzle.com/photopop_minimal_year_o...,PhotoPop Minimal Year Overlay Graduation Foil ...,84 people viewed this design,2026-02-11,"photopop graduation announcement, graduation a..."
21,https://rlv.zcache.com/photopop_modern_photo_c...,https://www.zazzle.com/photopop_modern_photo_c...,PhotoPop Modern Photo Collage Graduation Party...,225 people viewed this design,2026-02-11,"photopop, graduation announcement, graduation ..."
20,https://rlv.zcache.com/crushed_it_photopop_gra...,https://www.zazzle.com/crushed_it_photopop_gra...,Crushed It PhotoPop Graduation Party Foil Invi...,110 people viewed this design,2026-02-11,"graduation announcement template, photo gradua..."
18,https://rlv.zcache.com/contemporary_photopop_g...,https://www.zazzle.com/contemporary_photopop_g...,Contemporary PhotoPop Graduation Party Foil In...,153 people viewed this design,2026-02-11,"graduation announcement template, class of gra..."
16,https://rlv.zcache.com/modern_photopop_graduat...,https://www.zazzle.com/modern_photopop_graduat...,Modern PhotoPop Graduation Party Foil Invitation,294 people viewed this design,2026-02-11,"graduation announcement template, photo gradua..."
11,https://rlv.zcache.com/modern_photopop_collage...,https://www.zazzle.com/modern_photopop_collage...,Modern PhotoPop Collage Graduation Party Foil ...,519 people viewed this design,2026-02-11,"graduation party announcement, modern grad ann..."
15,https://rlv.zcache.com/photopop_modern_graduat...,https://www.zazzle.com/photopop_modern_graduat...,PhotoPop Modern Graduation Foil Invitation,181 people viewed this design,2026-02-10,"photopop, graduation announcement, photopop gr..."
1,https://rlv.zcache.com/modern_photopop_collage...,https://www.zazzle.com/modern_photopop_collage...,Modern PhotoPop Collage Graduation Party Invit...,5.1K people viewed this design,2026-02-10,"graduation party announcement, modern grad ann..."


In [20]:
# make the view column better, string to number
def string_to_int(text):
    match = re.search(r"\d+(\.\d+)?", str(text)) #search for the number
    
    if not match:
        return None
    
    number = float(match.group()) #extract the number
    
    if "K" in str(text): #if there any "K" in the string mulply the number with 1000
        return int(number * 1000)
    else:
        return int(number)

In [21]:
df['View']=df['View'].apply(string_to_int)

# End

## Making a Frequency Table for the Tags

In [22]:
tag_df = (
    df["Tag"]
    .str.split(", ") #split the tags from a single string to array of tags
    .explode() #flatten the tags, from array to each tags at a line
    .value_counts() #calculate the frequency count
)

In [29]:
tag_df.head(60)

Tag
senior photo announcement              14
high school graduation                 14
graduation announcement template       11
graduation announcement                11
photopop                               11
modern graduation design               10
photo graduation announcement          10
graduation party invite                10
college graduation                      9
college graduation announcement         9
photopop graduation                     8
photo graduation card                   8
modern grad announcement                6
graduation party announcement           5
senior photo template                   5
modern graduation template              4
high school graduation template         4
customizable graduation card            4
college graduation party                3
high school graduation party            3
minimalist graduation card              3
class of graduation                     3
graduation photo card                   3
editable graduation template  

In [24]:
#now using that frequency table assigning those frequency value to each tags on the dataframe also assign the total frequency count at the end of the each product's tag
tag_dict = tag_df.to_dict()
def tag_with_freq_and_total(tag_string):
    tags = [t.strip() for t in tag_string.split(",")] #converting string to list of tags
    counts = [tag_dict.get(tag, 0) for tag in tags] #getting the freq count for each tags and store it in a list
    tag_freq = [f"{tag} {count}" for tag, count in zip(tags, counts)] # assining the frequency count along with each tag
    total = sum(counts)
    return ", ".join(tag_freq), total   #returting the final string and total counts

In [25]:
df[["Tag_with_freq", "Total"]] = df["Tag"].apply(
    tag_with_freq_and_total
).apply(pd.Series)
df

,Image,Link,Title,View,Created Date,Tag,Tag_with_freq,Total
24,https://rlv.zcache.com/photopop_modern_arch_ph...,https://www.zazzle.com/photopop_modern_arch_ph...,PhotoPop Modern Arch Photo Graduation Party Fo...,265,2026-02-11,"graduation announcement template, photo gradua...","graduation announcement template 11, photo gra...",76
10,https://rlv.zcache.com/modern_vertical_photopo...,https://www.zazzle.com/modern_vertical_photopo...,Modern Vertical PhotoPop Graduation Party Foil...,169,2026-02-11,"graduation announcement, photopop graduation, ...","graduation announcement 11, photopop graduatio...",81
23,https://rlv.zcache.com/photopop_minimal_year_o...,https://www.zazzle.com/photopop_minimal_year_o...,PhotoPop Minimal Year Overlay Graduation Foil ...,84,2026-02-11,"photopop graduation announcement, graduation a...","photopop graduation announcement 1, graduation...",42
21,https://rlv.zcache.com/photopop_modern_photo_c...,https://www.zazzle.com/photopop_modern_photo_c...,PhotoPop Modern Photo Collage Graduation Party...,225,2026-02-11,"photopop, graduation announcement, graduation ...","photopop 11, graduation announcement 11, gradu...",82
20,https://rlv.zcache.com/crushed_it_photopop_gra...,https://www.zazzle.com/crushed_it_photopop_gra...,Crushed It PhotoPop Graduation Party Foil Invi...,110,2026-02-11,"graduation announcement template, photo gradua...","graduation announcement template 11, photo gra...",70
18,https://rlv.zcache.com/contemporary_photopop_g...,https://www.zazzle.com/contemporary_photopop_g...,Contemporary PhotoPop Graduation Party Foil In...,153,2026-02-11,"graduation announcement template, class of gra...","graduation announcement template 11, class of ...",72
16,https://rlv.zcache.com/modern_photopop_graduat...,https://www.zazzle.com/modern_photopop_graduat...,Modern PhotoPop Graduation Party Foil Invitation,294,2026-02-11,"graduation announcement template, photo gradua...","graduation announcement template 11, photo gra...",72
11,https://rlv.zcache.com/modern_photopop_collage...,https://www.zazzle.com/modern_photopop_collage...,Modern PhotoPop Collage Graduation Party Foil ...,519,2026-02-11,"graduation party announcement, modern grad ann...","graduation party announcement 5, modern grad a...",44
15,https://rlv.zcache.com/photopop_modern_graduat...,https://www.zazzle.com/photopop_modern_graduat...,PhotoPop Modern Graduation Foil Invitation,181,2026-02-10,"photopop, graduation announcement, photopop gr...","photopop 11, graduation announcement 11, photo...",60
1,https://rlv.zcache.com/modern_photopop_collage...,https://www.zazzle.com/modern_photopop_collage...,Modern PhotoPop Collage Graduation Party Invit...,5100,2026-02-10,"graduation party announcement, modern grad ann...","graduation party announcement 5, modern grad a...",44


In [26]:
df.to_excel("PhotoPop.xlsx", index=False) #save it to a exel file

In [16]:
df.to_csv("PhotoPop.csv") #save it to a csv file